# Fraud Lab: from transaction risk to review policy

This notebook explains the simulation experiment. All results are simulation-only. Run from `notebooks/`, or change the root path below. Use the project requirements in your notebook environment.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root / 'src'))
from train import simulate_transactions, chronological_split, run
import pandas as pd


## Explore the observable inputs
Labels are sampled after the observable features. The generator contains hidden risk, nonlinear interactions, and temporal drift. Velocity and typical amount fields are simulated pre-transaction aggregates; they are not derived from real account histories.

In [2]:
transactions = simulate_transactions()
transactions.head()

,Time,amount,account_age_days,transactions_1h,distance_km,new_device,new_merchant,hour,international,amount_to_median,failed_attempts_24h,Class
0,0.00,53.57,252,1,38.4,1,0,0,0,2.92,0,0
1,4.32,9.97,75,3,25.3,0,1,0,0,1.21,0,0
2,8.64,93.51,80,3,48.4,0,0,0,0,0.51,0,0
3,12.96,118.59,249,2,20.8,0,0,0,0,0.56,0,0
4,17.28,3.19,1050,2,3.6,0,0,0,0,0.67,0,0


In [3]:
pd.DataFrame({'rows': [len(transactions)], 'fraud': [transactions.Class.sum()], 'fraud_rate': [transactions.Class.mean()], 'missing_cells': [transactions.isna().sum().sum()]})

,rows,fraud,fraud_rate,missing_cells
0,60000,617,0.010283,0


In [4]:
transactions.groupby('Class')[['amount', 'new_device', 'transactions_1h', 'international']].mean()

,amount,new_device,transactions_1h,international
Class,,,,
0,80.221138,0.096678,1.289275,0.097974
1,82.742285,0.427877,2.085900,0.220421


## Split by time
Training is 60%, sigmoid calibration is 10%, validation is 10%, and test is 20%. Equal timestamps stay together. This protects the local fitting protocol from peeking at later periods; it does not validate the realism of the simulation.

In [5]:
parts = chronological_split(transactions)
pd.DataFrame([{'partition': name, 'rows': len(part), 'fraud': part.Class.sum(), 'start_hours': part.Time.min()/3600, 'end_hours': part.Time.max()/3600} for name, part in zip(['train','calibration','validation','test'],parts)])

,partition,rows,fraud,start_hours,end_hours
0,train,36000,326,0.0,43.1988
1,calibration,6000,72,43.2,50.3988
2,validation,6000,75,50.4,57.5988
3,test,12000,144,57.6,71.9988


## Train, calibrate, select, then evaluate
The complete implementation is in `src/train.py`. Both candidates use balanced training weights. Unweighted sigmoid calibration is separate. Selection uses validation recall at a 1% validation review target; AP breaks ties. Run the next cell to reproduce the experiment and regenerate the demo inputs.

In [6]:
report = run()

Source: synthetic; partitions: [36000, 6000, 6000, 12000]
logistic validation AP: 0.0990285049019529
boosting validation AP: 0.08848686245727627
{
  "selected_model": "logistic",
  "test_results": {
    "logistic": {
      "validation_ap": 0.0990285049019529,
      "validation_policy": {
        "tp": 9,
        "fp": 51,
        "fn": 66,
        "tn": 5874,
        "reviewed": 60,
        "precision": 0.15,
        "recall": 0.12,
        "review_rate": 0.01,
        "flagged_fraud_amount": 522.6,
        "missed_fraud_amount": 5100.26
      },
      "test_ap": 0.09174057363297924,
      "test_roc_auc": 0.7911752371045134,
      "test_brier": 0.011385941538216354,
      "threshold": 0.1138555455755964,
      "test_policy": {
        "tp": 22,
        "fp": 72,
        "fn": 122,
        "tn": 11784,
        "reviewed": 94,
        "precision": 0.23404255319148937,
        "recall": 0.1527777777777778,
        "review_rate": 0.007833333333333333,
        "flagged_fraud_amount": 1434.7

In [7]:
pd.DataFrame({name: {'validation_ap': result['validation_ap'], 'test_ap': result['test_ap'], 'test_roc_auc': result['test_roc_auc'], 'test_brier': result['test_brier'], 'test_recall': result['test_policy']['recall'], 'test_precision': result['test_policy']['precision'], 'test_review_rate': result['test_policy']['review_rate']} for name, result in report['models'].items()}).T

,validation_ap,test_ap,test_roc_auc,test_brier,test_recall,test_precision,test_review_rate
logistic,0.099029,0.091741,0.791175,0.011386,0.152778,0.234043,0.007833
boosting,0.088487,0.087354,0.760027,0.011480,0.118056,0.163462,0.008667


## Explore hypothetical review economics
This is a scenario calculation, not observed savings. Changing the test policy here is exploratory; do not report a newly optimized test cutoff as an unbiased result.

In [8]:
selected = report['models'][report['selected_model']]['test_policy']
review_cost, false_alert_cost, prevention_fraction = 3, 10, 0.8
net = selected['flagged_fraud_amount'] * prevention_fraction - selected['reviewed'] * review_cost - selected['fp'] * false_alert_cost
{'hypothetical_net_savings': round(net, 2), 'assumptions': {'review_cost': review_cost, 'false_alert_cost': false_alert_cost, 'prevention_fraction': prevention_fraction}}

{'hypothetical_net_savings': 145.82, 'assumptions': {'review_cost': 3, 'false_alert_cost': 10, 'prevention_fraction': 0.8}}

## Rerun on real data
Download the original ULB CSV using `python src/download_data.py`, then run `python src/train.py --source ulb --csv data/creditcard.csv`. That path has not yet been executed in the initial environment. The anonymous PCA features have unknown upstream fitting provenance. Read the README for data terms, limitations, and follow-up evaluation.

## Discussion
Why does a 1% validation review target produce a different review rate on test? Why is average precision more informative than accuracy here? What would delayed labels and repeated accounts change?